Ссылка на урок: [Спринт 2/6: 2 RAG: от сырых данных до точных ответов → Тема 2/4: Векторные базы и индексы → Урок 3/5: FAISS: индексы, скорость и компромиссы]( https://practicum.yandex.ru/learn/ai-engineer/courses/99ad0dee-d9e6-4da1-b921-64e7e9653a4e/sprints/1104074/topics/7c1be2cd-4a52-4f8a-8dce-db7d2e89a79d/lessons/5d6f1006-544b-49f4-af26-38a50457aeeb/#6f3d0d47-7b3e-45ce-9c05-5547bc720094 )
Вопрос: 

In [2]:
import numpy as np                                          # Матрицы эмбеддингов
from datasets import load_dataset                           # Скачивание корпуса MIRACL
from sentence_transformers import SentenceTransformer       # Эмбеддер на GPU

# --- Корпус: стримим русский MIRACL, берём пассажи разумной длины ---
# streaming=True не качает все 9.5 млн пассажей, а отдаёт их по одному
stream = load_dataset(
    "json",
    data_files="hf://datasets/miracl/miracl-corpus/"
               "miracl-corpus-v1.0-ru/docs-*.jsonl.gz",
    split="train", streaming=True)
texts: list[str] = []
for row in stream:
    t = row["text"].strip()
    if 100 <= len(t) <= 1000:          # Отсекаем слишком короткие и длинные
        texts.append(t)
    if len(texts) >= 100_100:
        break

CORPUS = texts[:100_000]               # 100 000 пассажей — корпус
QUERIES = texts[100_000:100_100]       # 100 пассажей — запросы
assert len(CORPUS) == 100_000 and len(QUERIES) == 100


def embed_e5_docs(model, texts):
    """Эмбеддинги документов через multilingual-e5: к каждому
    документу спереди добавляется префикс 'passage: '."""
    inputs = ["passage: " + t for t in texts]
    return model.encode(inputs, normalize_embeddings=True,
                        batch_size=64, show_progress_bar=True)


def embed_e5_queries(model, texts):
    """Эмбеддинги запросов через multilingual-e5: к каждому запросу
    спереди добавляется префикс 'query: '."""
    inputs = ["query: " + t for t in texts]
    return model.encode(inputs, normalize_embeddings=True,
                        batch_size=64, show_progress_bar=True)


# --- Считаем эмбеддинги на GPU (это самый долгий шаг, около 10–15 минут) ---
# FAISS требует непрерывный массив float32.
e5 = SentenceTransformer("intfloat/multilingual-e5-base", device="cuda")
corpus_vecs = np.ascontiguousarray(embed_e5_docs(e5, CORPUS), dtype="float32")
query_vecs = np.ascontiguousarray(embed_e5_queries(e5, QUERIES), dtype="float32")

# Сохраняем, чтобы не пересчитывать в Задании 6
np.save("corpus_vecs.npy", corpus_vecs)
np.save("query_vecs.npy", query_vecs)
d = corpus_vecs.shape[1]               # Размерность вектора e5-base: 768
print(f"Корпус: {corpus_vecs.shape}, запросы: {query_vecs.shape}")

/home/ubuntu/rag-vector-db/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RuntimeError: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver.